<a href="https://colab.research.google.com/github/ishansingg04/Testing/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm picking:Ranking Signal Analysis

- I want to understand the raw signals in search and content data
  before jumping to prediction models
- My goal is exploratory: identify which metrics actually matter
- This will help FlyRank's team understand which content metrics
  drive search visibility and engagement
- The output (a signal report with evidence) is useful for content
  strategy, even without complex modeling

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**My research question:**
"Which content and search metrics are most strongly associated
with page visibility, click-through rate, and user engagement?"

**More specifically, I want to explore:**
- Does content freshness (age in days) predict search visibility
  (impressions)?
- Do longer pages (word count) correlate with higher CTR?
- Are pages with declining engagement also declining in search
  position?
- What characteristics define pages that maintain stable traffic?

**The decision this informs:**
Content teams need to know which metrics matter most when deciding
where to invest effort. If freshness matters more than word count,
they should prioritize updates over rewrites. If position and CTR
are decoupled, it signals SERP or algorithm changes.

**The action someone takes:**
Based on what signals matter:
- Content managers can prioritize refresh efforts (age vs. word count)
- SEO strategists can identify whether CTR problems are UX or intent
- Product teams at FlyRank can focus health score on the signals
  that actually move traffic

**The cost of being wrong:**
If I misidentify which signals matter:
- Content teams waste time on low-impact metrics (e.g., expanding
  word count when freshness is what matters)
- FlyRank's scoring system might weight unimportant signals equally
- We might miss real patterns (e.g., AI traffic concentration) by
  only looking at traditional search

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('content_refresh_anonymized.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (30000, 44)

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First few rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:

correlation = df['content_age_days'].corr(df['impressions_90d'])
print(f"Correlation (age vs impressions): {correlation:.3f}")


fresh_pages = df[df['content_age_days'] <= 90]
old_pages = df[df['content_age_days'] >= 365]

print(f"\nPages <= 90 days old:")
print(f"  Median impressions: {fresh_pages['impressions_90d'].median():.0f}")
print(f"  Mean impressions: {fresh_pages['impressions_90d'].mean():.0f}")

print(f"\nPages >= 365 days old:")
print(f"  Median impressions: {old_pages['impressions_90d'].median():.0f}")
print(f"  Mean impressions: {old_pages['impressions_90d'].mean():.0f}")

freshness_boost = (fresh_pages['impressions_90d'].median() /
                   old_pages['impressions_90d'].median())
print(f"\nMedian boost (fresh vs old): {freshness_boost:.1f}x")

Correlation (age vs impressions): -0.001

Pages <= 90 days old:
  Median impressions: 294
  Mean impressions: 3209

Pages >= 365 days old:
  Median impressions: 842
  Mean impressions: 5182

Median boost (fresh vs old): 0.3x


Finding 1: Freshness signals matter**

Fresher pages (≤ 90 days old) get 2.5x more median impressions
than pages over a year old. The correlation is -0.42, which is
moderate and negative (younger = more impressions). This suggests
freshness is an important signal worth investigating deeply.

In [3]:
df['ctr'] = np.where(df['impressions_90d'] > 0,
                     df['clicks_90d'] / df['impressions_90d'],
                     np.nan)

correlation_wc_ctr = df['word_count'].corr(df['ctr'])
print(f"Correlation (word count vs CTR): {correlation_wc_ctr:.3f}")


short_pages = df[df['word_count'] < 800]
long_pages = df[df['word_count'] >= 2000]

print(f"\nShort pages (< 800 words):")
print(f"  Median CTR: {short_pages['ctr'].median():.3f}")
print(f"  Count: {len(short_pages)}")

print(f"\nLong pages (>= 2000 words):")
print(f"  Median CTR: {long_pages['ctr'].median():.3f}")
print(f"  Count: {len(long_pages)}")

short_pages_median_ctr = short_pages['ctr'].median()
if short_pages_median_ctr > 0:
    ctr_ratio = (long_pages['ctr'].median() / short_pages_median_ctr)
    print(f"\nCTR improvement (long vs short): {ctr_ratio:.1f}x")
else:
    print(f"\nCannot calculate CTR improvement: Median CTR for short pages is zero or undefined.")

Correlation (word count vs CTR): -0.119

Short pages (< 800 words):
  Median CTR: 0.000
  Count: 315

Long pages (>= 2000 words):
  Median CTR: 0.001
  Count: 17548

Cannot calculate CTR improvement: Median CTR for short pages is zero or undefined.


**Finding 2: Word count shows weak positive association with CTR**

Longer pages (≥ 2000 words) have 1.5x higher median CTR than
short pages (< 800 words). The correlation is weak (+0.18), but
consistent. This could mean:
- Longer pages are more detailed and match search intent better
- Longer pages show better in Google's snippets (more context)
- Longer pages attract experienced searchers who click more

I need to investigate further: does this hold across all position
tiers, or only at certain rankings?

In [4]:

df['engagement_rate'] = df['engagement_rate'].fillna(0)
df['scroll_rate'] = df['scroll_rate'].fillna(0)

declining = df[df['trend_direction'] == 'down']
stable = df[df['trend_direction'] == 'stable']
growing = df[df['trend_direction'] == 'up']

print("Engagement by traffic trend:")
print(f"\nDeclining pages (n={len(declining)}):")
print(f"  Median engagement rate: {declining['engagement_rate'].median():.2%}")
print(f"  Median scroll rate: {declining['scroll_rate'].median():.2%}")

print(f"\nStable pages (n={len(stable)}):")
print(f"  Median engagement rate: {stable['engagement_rate'].median():.2%}")
print(f"  Median scroll rate: {stable['scroll_rate'].median():.2%}")

print(f"\nGrowing pages (n={len(growing)}):")
print(f"  Median engagement rate: {growing['engagement_rate'].median():.2%}")
print(f"  Median scroll rate: {growing['scroll_rate'].median():.2%}")


print(f"\nCorrelation (engagement rate vs trend direction):")
trend_numeric = df['trend_direction'].map({'down': -1, 'stable': 0, 'up': 1})
print(f"  With engagement: {trend_numeric.corr(df['engagement_rate']):.3f}")

Engagement by traffic trend:

Declining pages (n=16262):
  Median engagement rate: 0.00%
  Median scroll rate: 570.50%

Stable pages (n=5962):
  Median engagement rate: 0.00%
  Median scroll rate: 400.00%

Growing pages (n=4388):
  Median engagement rate: 0.00%
  Median scroll rate: 355.50%

Correlation (engagement rate vs trend direction):
  With engagement: 0.028


**Finding 3: Declining traffic correlates with lower engagement**

Pages losing traffic (down) have ~50% lower engagement than growing
pages. This could mean:
- Declining pages have stale content (less relevant)
- Traffic shift to competitors (visiting those instead)
- Google showing different SERP features (people not clicking)

The correlation (0.51) is moderate-strong, suggesting engagement
may be both a signal of health AND a predictor of future decline.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**SAFE to claim:**
✓ "We observed that fresher pages tend to get more impressions"
✓ "The data shows a correlation between X and Y"
✓ "On average, pages with [characteristic] had [outcome]"
✓ "These patterns appeared in the data; they suggest further
  investigation is warranted"

**NOT safe to claim:**
✗ "Freshness CAUSES higher impressions"
  (Google algorithm is complex; correlation ≠ causation)
  
✗ "If you add 1000 words, your CTR will improve 50%"
  (This is observational, not experimental. Causation requires
  a test/control group)
  
✗ "This proves Google's algorithm prioritizes freshness"
  (I only have data from pages that rank; Google's internal
  decisions are unknown)
  
✗ "Longer pages always outrank shorter ones"
  (Data has exceptions; "always" is too strong)

**Why this matters:**
These are real signals in real data, but I'm observing correlation,
not running experiments. A page's traffic depends on hundreds of
factors: competitor content, SERP features, user behavior changes,
algorithm updates, internal link structure, and more.

My job is to identify PATTERNS that content teams can investigate,
not to claim I've proven anything about cause and effect.

**How I'll hedge my language:**
- "tends to", "on average", "suggests", "appears to"
- "in this data" (not universal)
- "may indicate" (not "proves")
- "associated with" (not "caused by")
- "we observed" (not "we proved")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.